<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Meet the Larkfield images

**[Meet the Larkfield images](https://learning.nextia-ai.com/courses/vision/m01/meet-the-larkfield-images/)** · Vision AI: See, Count and Inspect · Module 1, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** open the Larkfield image sets, read their manifests and licences, look at a photo as numbers, and read the comparison table that every module of the course fills.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no graphics card. |
| **You should know** | The five kinds of answer and images as numbers, from [Five kinds of answer](https://learning.nextia-ai.com/courses/vision/m01/five-kinds-of-answer/) and [Images are numbers](https://learning.nextia-ai.com/courses/vision/m01/images-are-numbers/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m01/meet-the-larkfield-images/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M01-L04-meet-the-larkfield-images/meet-the-larkfield-images-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 39 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "archives/returns.zip": "dcf02f521f68228d34717b303628b2db8b19d18433126da2ce64047f1449210f",
    "archives/pallets.zip": "cec4b211fa4e434936fc0c1980accfc74392f1ced9de6c04b5421b9ec6af56e3",
    "archives/customer_photos.zip": "dba0cbb32bbf3d3865a48c29ffcbc3e8c09ef8c7485296d304e6a338a803f67d",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The catalogue

Each row of `data/catalogue/manifest.csv` is one product photo: its file, its split (`train`, `val` or `test`), its label, and where the photo comes from (`source_url`, `licence`, `author`).

> **Predict.** The catalogue has eight categories. Do they all have the same number of training photos?

In [ ]:
catalogue = pd.read_csv("data/catalogue/manifest.csv")
print(catalogue["split"].value_counts().to_string())
print(catalogue[catalogue["split"] == "train"]["label"].value_counts().to_string())

> **Check.** You should see 453 training, 120 validation and 160 test photos. The candle category has only 15 training photos: a problem we planted. Module 3 comes back to it.

Now look at two photos of each category. When you show a photo in your own work, name its author and the licence: the manifest has both.

In [ ]:
for label in CLASSES[:4]:
    rows = catalogue[(catalogue["split"] == "train") & (catalogue["label"] == label)].head(4)
    show("data/" + rows["file"], [f"{label}\n{a[:18]}" for a in rows["author"]])

## 2. A photo is a grid of numbers

A catalogue photo is 256 × 256 pixels, and each pixel has three numbers: red, green and blue, from 0 to 255.

In [ ]:
path = "data/" + catalogue.iloc[0]["file"]
pixels = np.asarray(Image.open(path))
print(pixels.shape, pixels.dtype)
print("the pixel in the middle:", pixels[128, 128], " mean of each channel:", pixels.reshape(-1, 3).mean(0).round(1))

> **Check.** The shape is `(256, 256, 3)`: rows, columns and the three channels.

## 3. The returns, the pallets and the customer photos

The returned candles (`data/returns/`) are good or damaged. A damaged photo has a mask: white where the damage is. The pallets (`data/pallets/`) are drawn by our code, so every box is known exactly.

In [ ]:
returns = pd.read_csv("data/returns/manifest.csv")
print(returns.groupby(["split", "status"]).size().unstack().to_string())
damaged = returns[(returns["split"] == "train") & (returns["status"] == "damaged")].iloc[0]
show(["data/" + damaged["file"], "data/" + damaged["mask_file"]], [damaged["damage"], "the mask"], size=3.2)

In [ ]:
pallets = json.load(open("data/pallets/test.json"))
scene = pallets["images"][2]
boxes = [a["bbox"] for a in pallets["annotations"] if a["image_id"] == scene["id"]]
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.imshow(Image.open("data/" + scene["file_name"])); ax.axis("off")
for x, y, w, h in boxes:
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="red", lw=1))
ax.set_title(f"{scene['scene_id']}: {scene['count']} boxes, {scene['crowding']}"); plt.show()

The customer photos are phone photos of the catalogue test products, made by our code from the catalogue photo: another angle, poor light, blur. Module 7 matches them to the catalogue.

In [ ]:
customers = pd.read_csv("data/customer_photos/manifest.csv")
row = customers.iloc[0]
show(["data/" + row["file"], "data/" + catalogue.set_index("image_id").loc[row["catalogue_image_id"], "file"]],
     ["customer photo", "catalogue photo"], size=3.2)

## 4. The cost table and the comparison table

Hamid's costs are **assumptions** (`data/costs.json`). The comparison table (`data/comparison_table.json`) holds the course's measured rows: every module adds to it, on the same photos.

In [ ]:
print(json.load(open("data/costs.json")))
table = cp.Table.load("data/comparison_table.json")
table.show("catalogue", columns=("method", "setting", "value", "worst", "worst_value", "model_mb", "labelled_images"))

> **Check.** The first rows are the simple methods (rules, a colour histogram, HOG) with an accuracy under 0.4. The pretrained models are near 0.9.

## 5. The hidden test

The catalogue, returns and pallets each have a **hidden test set** that is not in this folder. The course team keeps it, so that a final score is honest: you choose a method and its thresholds on `train` and `val`, and you measure the chosen one once.

## 6. Your turn: the damaged training photos

> **Your turn.** Count the damaged returns photos in the **training** split, by damage type. A photo can have two types (`"; "` between them): count each type. Put the result in `damage_counts`, a dictionary from type to count.

In [ ]:
from collections import Counter
damage_counts = ...  # for example {"chunk of wax missing": 3, ...}

In [ ]:
expect_hash('damage_counts', damage_counts, 'c5d37216c62b7fb2')

## 7. Failure case: a photo that is not what its label says

Real labels are noisy. The Open Images "Lamp" class has a few carved pumpkins, and "Mug" has beer glasses. The next cell shows some lamp photos. Would your lamp model learn from a pumpkin?

In [ ]:
rows = catalogue[catalogue["label"] == "lamp"].iloc[20:28]
show("data/" + rows["file"], list(rows["image_id"]))

## Reset

If you change files in `larkfield-images` and want to start again, delete the folder and run the setup cells again. In the project on your computer, `vision reset` checks every data file.

## Next

You opened the Larkfield images and the comparison table. The [lesson page](https://learning.nextia-ai.com/courses/vision/m01/meet-the-larkfield-images/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Rules and colour histograms](https://learning.nextia-ai.com/courses/vision/m02/rules-and-colour-histograms/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m01/meet-the-larkfield-images/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.